In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import datetime
import sys

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_bar=-0.05):
        self.phi0 = phi0
        print(f'Initial polymer volume fraction = {phi0:.2f} [dimensionless];', end=' ')
        self.mu_bar = mu_bar   # 🔥 NUEVO
        print(f'Normalized chemical potential = {mu_bar:.6f} [dimensionless?];', end=' ')
        #
        # Values for T and V_m taken from p.1584 in Kang & Huang, JMPS 58 (2010)
        T = 25+273.15   # 25ºC, in [K]
        K_B = 1.380649e-23  # in m^2*kg*s^{-2}*K^{-1}, i.e. in N*m/K
        V_m = 3e-29     # volume of a molecule of solvent, in this case water, in m^3
        # self.entropic_unit = 136.6  # measured in MPa
        self.entropic_unit = K_B*T/V_m*1e-6  # measured in MPa
        print(f'Entropic unit = {self.entropic_unit:.2f} [MPa];', end='\n')
        #
        # self.G = 0.13               # measured in MPa
        # self.gamma = self.G/self.entropic_unit
        self.gamma = 0.001           # As in the simulations of Sect. 5 in Kang & Huang JMPS 2010
        # self.chi =  0.348
        self.chi = 0.4
        self.G = self.gamma*self.entropic_unit
        print(f'gamma=N*V_m = {self.gamma:.2e} [dimensionless];', end=' ')
        print(f'Shear modulus = N*K_B*T = {self.G:.2f} [MPa];', end=' ')
        print(f'Flory parameter = {self.chi:.3f} [dimensionless];', end='\n')
        #
        vapor_pressure = 3.2e-3    # 3.2 KPa, but measured in MPa
        self.p0_bar = vapor_pressure/(self.entropic_unit)
        print(f'Normalized vapor pressure = {self.p0_bar:.2e} [dimensionless];', end=' ')
        self.p_bar = self.p0_bar * np.exp(mu_bar)
        print(f'External solvent pressure = {self.p_bar*self.entropic_unit*1e3:.2f} [KPa];', end=' ')
        print(f'Normalized external pressure = {self.p_bar:.2e} [MPa];', end='\n')
        #        
        # self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm

        self.filename_suffix = f'_phi0={self.phi0:.1f}_muBarAbs={np.abs(mu_bar):.6f}'
        print(f'Filename suffix: ' + self.filename_suffix, end='\n')

        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxIsotropic(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, lambda_initial)[0]
        print(f'Isotropic extension: {self.lambda_iso:.3f}; lambda_initial = {lambda_initial}', end=' ')

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxUniaxial(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, lambda_initial)[0]
        print(f'Uniaxial extension: {self.lambda_target:.3f}; lambda_initial = {lambda_initial}', end='\n')

        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit; gamma=gel.gamma; mu_bar=gel.mu_bar; p_bar=gel.p_bar
            J= lambda1*lambda2*lambda3
            phi = phi0/J
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2 - 3) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi) - gamma*log(J) + (p_bar - mu_bar)*(J-phi0) )

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                
        print(f'Energy density of isotropic expansion: {self.reference_energy_density:.5f}', end=' ')

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J)) - self.gamma*log(J) + (self.p_bar - self.mu_bar)*(J-self.phi0)

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2  - self.gamma/J +self.p_bar - self.mu_bar

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C_tensor = F.trans * F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit               
        
        reference_energy_density =  self.reference_energy_density
        
        return 0.5*G*(Trace(C_tensor) - 3) + nu*gel.H(J) - reference_energy_density

In [2]:

# ===================== MU-BASED LOADING (MODIFICADO) =====================
import numpy as np

def mu_fun(lambda_val, Nv, w):
    return (
        np.log(1.0 - 1.0 / lambda_val)
        + 1.0 / lambda_val
        + w / (lambda_val ** 2)
        + Nv * (lambda_val - 1.0 / lambda_val)
    )

def Solve_incremental_softening(gel, SolveNonlinearMinProblem,
                                lambda_start=1.1,
                                lambda_target=2.0,
                                n_steps=16):

    Nv = getattr(gel, "Nv", 0.001)
    w  = getattr(gel, "w", 0.4)

    lambdas = np.linspace(lambda_start, lambda_target, n_steps)
    mus = np.array([mu_fun(l, Nv, w) for l in lambdas])

    results = []

    print("\n=== Incremental solve en μ ===\n")

    for i, (lambda_val, mu) in enumerate(zip(lambdas, mus)):

        print(f"[Paso {i+1}/{n_steps}] λ={lambda_val:.4f}, μ={mu:.6f}")

        gel.mu_bar = mu   # ← control principal

        # NO cambiar gamma
        # gel.gamma = constante

        if hasattr(gel, "lambda_h"):
            gel.lambda_h = lambda_val

        if hasattr(gel, "update_internal_state"):
            gel.update_internal_state()

        sol = SolveNonlinearMinProblem(gel)

        results.append({
            "lambda": lambda_val,
            "mu": mu,
            "solution": sol
        })

    return results
# ========================================================================


In [3]:
# experimental acceleration
def SolveNonlinearMinProblem(a, gfu, tol=1e-08, maxits=50, alpha=5e-2):
    
    start_time = datetime.datetime.now()  

    res = gfu.vec.CreateVector()
    du  = gfu.vec.CreateVector()
    
    # precond local, multigrid, bddc
    precond = 'bddc'
    c = Preconditioner(a, precond)

    for it in range(maxits):

        with TaskManager():
            a.Apply(gfu.vec, res)
            c = Preconditioner(a, precond)
            a.AssembleLinearization(gfu.vec)
            
            c.Update()
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            du.data = alpha * inv * res

            # update
            gfu.vec.data -= du

        # stopping criteria
        stopcritval = sqrt(abs(InnerProduct(du, res)))

        print("Newton iteration:", it, "Time elapsed =" + str(datetime.datetime.now() - start_time))
        print("<A u", it, ", A u", it, ">_{-1}^0.5 = ", stopcritval)

        if stopcritval < tol:
            break

    return gfu, stopcritval, it

In [4]:

# data = [dummy, L, w, d, phi0, abs(mu_bar)]   We will suppose that mu_bar is negative
data = ['','90', '15.0', '1.62', '1', '0.0916'] 
order = 1

mesh_file = 'meshes/mesh0.vol.gz'

L = float(data[1])
d = float(data[2])
w = float(data[3])
phi0 = float(data[4])
mu_bar = - float(data[5])

print(f'L={L}, w={w}, d={d}, phi0={phi0}, mu_bar={mu_bar}')

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_bar=mu_bar)

modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Space()

L=90.0, w=1.62, d=15.0, phi0=1.0, mu_bar=-0.0916
Initial polymer volume fraction = 1.00 [dimensionless]; Normalized chemical potential = -0.091600 [dimensionless?]; Entropic unit = 137.21 [MPa];
gamma=N*V_m = 1.00e-03 [dimensionless]; Shear modulus = N*K_B*T = 0.14 [MPa]; Flory parameter = 0.400 [dimensionless];
Normalized vapor pressure = 2.33e-05 [dimensionless]; External solvent pressure = 2.92 [KPa]; Normalized external pressure = 2.13e-05 [MPa];
Filename suffix: _phi0=1.0_muBarAbs=0.091600
Isotropic extension: 1.262; lambda_initial = 1.2700000000000002 Uniaxial extension: 2.000; lambda_initial = 2.0100000000000007
Energy density of isotropic expansion: -55.06968 

NameError: name 'Solve_gel3d' is not defined

In [5]:

# ===================== MU-BASED LOADING (MODIFICADO) =====================
import numpy as np

def mu_fun(lambda_val, Nv, w):
    return (
        np.log(1.0 - 1.0 / lambda_val)
        + 1.0 / lambda_val
        + w / (lambda_val ** 2)
        + Nv * (lambda_val - 1.0 / lambda_val)
    )

def Solve_incremental_softening(gel, SolveNonlinearMinProblem,
                                lambda_start=1.1,
                                lambda_target=2.0,
                                n_steps=16):

    Nv = getattr(gel, "Nv", 0.001)
    w  = getattr(gel, "w", 0.4)

    lambdas = np.linspace(lambda_start, lambda_target, n_steps)
    mus = np.array([mu_fun(l, Nv, w) for l in lambdas])

    results = []

    print("\n=== Incremental solve en μ ===\n")

    for i, (lambda_val, mu) in enumerate(zip(lambdas, mus)):

        print(f"[Paso {i+1}/{n_steps}] λ={lambda_val:.4f}, μ={mu:.6f}")

        gel.mu_bar = mu   # ← control principal

        # NO cambiar gamma
        # gel.gamma = constante

        if hasattr(gel, "lambda_h"):
            gel.lambda_h = lambda_val

        if hasattr(gel, "update_internal_state"):
            gel.update_internal_state()

        sol = SolveNonlinearMinProblem(gel)

        results.append({
            "lambda": lambda_val,
            "mu": mu,
            "solution": sol
        })

    return results
# ========================================================================
